In [40]:
# Strat Data - 1-July-2025
# Purpose: Contract Deliverable for Bonsai Informatica. This application will prompt the 
#          user for the name of a data set to load and will then allow the user to search
#          for a word in the second data field. This application satisfies the following
#          requirements from the customer:
#
# Application Requirements:
#     1. The application must be able to read a data set from a file where the format is 
#        consistent with the provided data record format.
#     2. The application must be able to retrieve a record or records from the file based 
#        on provided search criteria.
#     3. When multiple records are returned, they must be displayed in ascending order of 
#        the TLEN field in the record.
#     4. When displaying a record, only the TLEN field and both data fields should be displayed.  
#     5. When displaying a record, each field should be separated by a tab.
#     6. The application must be able to accept a word to search for the word in the second data field. 
#     7. When searching, if the word is not found in the second data field, the application must 
#        indicate if there are any records where the word was found in the first data field. 
#     8. The application must provide an indication of how many records were searched. 
#     9. The application must provide an indication of how many records were found that 
#        matched the search criteria. 
#    10. The application must display all records containing relevant information separated by 
#        header information.
#    11. The application must be optimized to minimize the time necessary to perform the search function.
#
# This application assumes that the first line of each data file is a header line that will be
# skipped when creating valid records.
#
# This application supports the following data record format:
#
# Each data record has five pipe-delimited fields, with each field as follows: 
#     1. sequence – a zero-based numerical sequence indicating the position of the record 
#                   in the data file. 
#     2. TLEN – a positive integer representing the length of both data fields without any delimiters. 
#     3. data-a – a single word 
#     4. data-b – a single word 
#     5. validation – a SHA-256 checksum of the four previous fields inclusive of field-separating 
#                     characters between the four fields.
#
#
# Version               Author         Date              Description
#######################################################################################################
#    3                  fjm            8-Jul-2025        Fixed bug with hash table creation
#    2                  fjm            1-Jul-2025        Improved hash function
#    1                  fjm            1-Jul-2025        Initial Code
#######################################################################################################

##
## Import csv module to make working with the data file easier
##
import csv
# --- ADDED FOR TEST PLAN ---
import time

##
## Create a Python class to hold the data from each record
##
class datarecord:
    ##
    ## Class constructor
    ##
    ## Parameters:
    ##     sequence - sequence number of the record in the data file
    ##     TLEN - Length of dataA + dataB
    ##     dataA - A single word
    ##     dataB - A single word
    ##     validation - SHA-256 validation string for the first four
    ##                  fields in the record
    ##
    def __init__(self,sequence,TLEN,dataA,dataB,validation):
        self.sequence = sequence
        self.TLEN = TLEN
        self.dataA = dataA
        self.dataB = dataB
        self.validation = validation


    ##
    ## printHeader - function to Print Header information for records
    ##
    def printHeader(self):
        print("TLEN\tdata-a\tdata-b\n")

              
    ##
    ## printRecord - function to Print the record in the manner the 
    ## application requires. TLEN and then Both Data fields with information
    ## separated by tabs.
    ##
    def printRecord(self):
        print(f"{self.TLEN}\t{self.dataA}\t{self.dataB}\n")


    ##
    ## hashFunction - function to calculate the hash value for this specific
    ## datarecord object. This hash function creates a value from the
    ## characters in dataB and then returns the integer
    ## remainder of that value divided by tableSize. This will facilitate 
    ## searching for values in dataB.
    ##
    ## Parameters:
    ##      tableSize - variable indicating the size of the Hash Table. This
    ##                  must be indicated as an positive integer
    ##
    ## Returns:
    ##      positive integer indicating the index into the Hash Table of 
    ##      size tableSize.
    ##
    def hashFunction(self,tableSize):
        source = self.dataB
        length = len(source)
        simpleSum = 0

        for char in source:
            simpleSum = simpleSum + ord(char)

        # Return the value adjusted for the correct Hash Table size
        return simpleSum % tableSize

    ##
    ## extHashIndex - function to calculate the hash value for this provided
    ## string. This hash function creates a value from the
    ## characters in the data parameter and then returns the integer
    ## remainder of that value divided by tableSize. This will facilitate 
    ## searching for values in a given sized Hash Table.
    ##
    ## Parameters:
    ##      data - The string to be hashed
    ##      tableSize - variable indicating the size of the Hash Table. This
    ##                  must be indicated as an positive integer
    ##
    ## Returns:
    ##      positive integer indicating the index into the Hash Table of 
    ##      size tableSize.
    ##
    def extHashIndex(data,tableSize):
        source = data
        simpleSum = 0

        for char in source:
            simpleSum = simpleSum + ord(char)

        return simpleSum % tableSize

    ##
    ## secondaryHashFunction - function to calculate the hash value for dataA for
    ## this specific datarecord object. This hash function creates a value from the
    ## characters in dataA, and then returns the integer
    ## remainder of that value divided by tableSize.
    ##
    ## Parameters:
    ##      tableSize - variable indicating the size of the Hash Table. This
    ##                  must be indicated as an positive integer
    ##
    ## Returns:
    ##      positive integer indicating the index into the Hash Table of 
    ##      size tableSize.
    ##
    def secondaryHashFunction(self,tableSize):
        source = self.dataA
        simpleSum = 0

        for char in source:
            simpleSum = simpleSum + ord(char)

        # Return the value adjusted for the correct Hash Table size
        return simpleSum % tableSize

        
    ##
    ## readDataFile - function to read a pipe delimited data file and return
    ## a list of datarecord object. This method assumes that the first line of
    ## the file is a header line.
    ##
    ## Parameters:
    ##     fileName - the name of the datafile to read from
    ##
    ## Returns:
    ##     list of datarecord objects in the same order as they appeared in
    ##     the data file.
    def readDataFile(fileName):
        ## define our output list
        records = []

        ## read data file
        with open(fileName,'r') as file:
            csvReader = csv.reader(file, delimiter='|')

            ## get the headers - this is used as a no-op as the headers
            ## aren't used here.
            header = next(csvReader)



            ## Read the file and create datarecord objects, add them to the list
            for row in csvReader:
                records.append(datarecord(row[0],row[1],row[2],row[3],row[4]))

            ## close file
            file.close()

        return records

        
    ##
    ## matches - function to determine whether or not this record matches
    ## a specific search criteria.
    ##
    ## Parameters:
    ##      selector - boolean value to indicate which field to match against.
    ##                 if the value is true, matching is performed against 
    ##                 dataB. If the value is false, matching is performed
    ##                 against dataA.
    ##      value - the value to test against
    ## 
    ## Returns:
    ##      True if the record match is successful, False if the record
    ##      does not match the search criteria
    def matches(self,selector,value):
        if selector:
            return (self.dataB == value)
        else:
            return (self.dataA == value)
            

##
## promptUser - function used to prompt the user for input. They will either
## enter a one word search string or a two word command to quit. 
##
## Returns:
##      String containing user input.
##
def promptUser():
    print("Please enter a single word to be located within the data set.")
    print("If you wish to quit, please enter: I quit\n")
    out = input("Search string: ")
    return out

##
## conductSearch - function used to conduct the records search. This function
## will first search dataB, and if no records are present that match the search
## criteria, will then search dataA.
##
## Parameters:
##      search - the string to be located
##      primaryHT - The primary Hash Table indexed by dataB
##      altHT - The alternate Hash Table indexed by dataA
##
## Returns:
##      this function returns no data, but will display formatted results to the screen.
##
def conductSearch(search, primaryHT, altHT):
    # --- ADDED FOR TEST PLAN ---
    if TIME:
        start_time = time.time()
    
    ## Calculate index - because both hash tables are the same size, we only
    ## need to do this once
    index = datarecord.extHashIndex(search,len(primaryHT))
    if BETTER_HASH:
        index = alternateHash(search, len(primaryHT))

    ## Flag to determine if we need to check the alternate Hash Table at all.
    checkAlt = False
    
    ## accumulator variable for how many records were checked.
    checked = 0

    ## display list
    matchingRecords = []
    
    ## Check Primary Hash Table
    bucket = primaryHT[index]

    if len(bucket) == 0:
        ## Nothing Here, check alternate Hash Table
        checkAlt = True
    else:
        for i in bucket:
            checked = checked + 1
            if i.matches(True,search):
                matchingRecords.append(i)
        if len(matchingRecords) == 0:
            ## Nothing in primary, check alternate Hash Table
            checkAlt = True

    ## If True, check alternate table
    if checkAlt:
        bucket = altHT[index]
        if len(bucket) == 0:
            ## Nothing Here, we are done
            pass
        else:
            for i in bucket:
                checked = checked + 1
                if i.matches(False,search):
                    matchingRecords.append(i)

    ## All checks have been completed, so we need to output our results.
    if not checkAlt:
        print(f"{len(matchingRecords)} matching records were found in dataB")
        matchingRecords[0].printHeader()
        for i in matchingRecords:
            i.printRecord()
    else:
        print("No matching records were found in dataB")
        if len(matchingRecords) == 0:
            print("No matching records were found in dataA")
        else:
            print(f"{len(matchingRecords)} matching records were found in dataA")
            matchingRecords[0].printHeader()
            for i in matchingRecords:
                i.printRecord()

    ## Provide additional data
    print(f"{checked} records were checked while executing this search.")     

    # --- ADDED FOR TEST PLAN ---
    if TIME:
        elapsed_time = (time.time() - start_time) * 1e6
        print(f"Runtime for search: {elapsed_time:.2f} microseconds")

# --- ADDED FOR TEST PLAN ---
# alternate hash function using the built in method to distribute values
# across buckets more effectively
def alternateHash(data, tableSize):
    return hash(data) % tableSize
        

## 
## main program
##
if __name__ == '__main__':
    ##
    ## Configure DEBUG flag. True = print DEBUG information, False = be quiet...
    ##
    DEBUG = False

    # --- ADDED FOR TEST PLAN ---
    # configure TIME flag, true to measure and display run time for searches
    TIME = True
    # configure BETTER_HASH flag to use an alternate hash function that should improve performance
    BETTER_HASH = True

    ##
    ## Get name of data file from user
    ## 
    dataFile = input("Please enter the name of the data file to process: ")

    if DEBUG:
        ## Print the name of the data file used
        print(f"File containing data set: {dataFile}")

    ##
    ## Read records into a list
    ##
    records = datarecord.readDataFile(dataFile)

    if DEBUG:
        ## Print number of records retrieved
        print(f"{len(records)} records retrieved.\n")

        # --- ADDED FOR TEST PLAN ---
        for i in range(0, 5):
            it = records[i]
            print(f"seq: {it.sequence} TLEN: {it.TLEN} dataA: {it.dataA} dataB: {it.dataB} val: {it.validation}")

    ##
    ## Create Empty Hash Tables.
    ##
    hashTable = []
    altHashTable = []
    ## for i in records:
    for i in range(10000):
        tmpA = []
        tmpB = []
        hashTable.append(tmpA)
        altHashTable.append(tmpB)

    if DEBUG:
        ## Print size of Hash Table
        print(f"Hash Table size = {len(hashTable)} elements.\n")

        ## Print Hash Table
        print(hashTable)

    ## 
    ## Populate the Hash Tables to promote fast searching
    ## 
    difference = 0
    primarySize = len(hashTable)
    alternateSize = len(altHashTable)
    for i in records:
        index = i.hashFunction(primarySize)

        if BETTER_HASH:
            index = alternateHash(i.dataB, primarySize)
        hashTable[index].append(i)

        altIndex = i.secondaryHashFunction(alternateSize)
        if BETTER_HASH:
            altIndex = alternateHash(i.dataA, alternateSize)
        altHashTable[altIndex].append(i)

        if index != altIndex:
            difference = difference + 1
            
        if DEBUG:
            pass
            ## Print Entry
            ## i.printRecord()

    if DEBUG:
        print(f"{difference} different hash table indexes across {len(records)} records.")
        ## Calculate the number of empty buckets and the bucket with the largest size
        empty = 0
        maxLen = 0
        for i in range(primarySize):
            entries = len(hashTable[i])
            if entries == 0:
                empty = empty + 1
            elif entries > maxLen:
                ## print(f"index: {i}\t{hashTable[i]}")
                maxLen = entries
        print("Primary Hash Table")
        print(f"Empty buckets: {empty} out of {primarySize}")
        print(f"Maximum bucket size: {maxLen}")
        
        altEmpty = 0
        altMaxLen = 0
        for i in range(alternateSize):
            entries = len(altHashTable[i])
            if entries == 0:
                altEmpty = altEmpty + 1
            elif entries > altMaxLen:
                altMaxLen = entries
        print("Secondary Hash Table")
        print(f"Empty buckets: {altEmpty} out of {alternateSize}")
        print(f"Maximum bucket size: {altMaxLen}")

    ##
    ## Main loop
    ##
    exit = False
    while not exit:
        search = promptUser()

        ## Check for exit condition
        if len(search.split()) != 1:
            exit = True
            print("Exiting...")
        else:
            ## Run search
            print(f"Searching for {search}...")
            conductSearch(search,hashTable,altHashTable)


Please enter the name of the data file to process:  data/set5.csv


Please enter a single word to be located within the data set.
If you wish to quit, please enter: I quit



Search string:  staining


Searching for staining...
1 matching records were found in dataB
TLEN	data-a	data-b

16	chymosin	staining

9 records were checked while executing this search.
Runtime for search: 154.26 microseconds
Please enter a single word to be located within the data set.
If you wish to quit, please enter: I quit



Search string:  i quit


Exiting...
